## BUSI Breast Ultrasound Classification

Breast ultrasound image classification into **benign**, **malignant**, and **normal** classes using transfer learning on the BUSI dataset.

### Dataset
**780 images** across three classes:
* Benign: 437
* Malignant: 210
* Normal: 133

### Model
* **Base model:** ResNet50 pretrained on ImageNet
* **Approach:** Transfer learning
* **Fine-tuning:** Last 30 layers unfrozen and fine-tuned

### Results

| Configuration                 | Accuracy | Malignant Recall |
| ----------------------------- | -------: | ---------------: |
| Before fine-tuning            |  **83%** |          **78%** |
| After fine-tuning (30 layers) |  **88%** |          **91%** |

Fine-tuning improved overall accuracy from **83% to 88%** and increased **malignant recall from 78% to 91%**, improving the model's ability to correctly identify malignant cases.


### Dataset Preparation

In [1]:
import os
import shutil
import numpy as np 
# Correct source and destination paths
input_path = "/kaggle/input/datasets/aryashah2k/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"
working_path = "/kaggle/working/Dataset_BUSI_with_GT"

# Step 1: Remove any wrong file or folder at working_path
if os.path.exists(working_path):
    if os.path.isdir(working_path):
        shutil.rmtree(working_path)
    else:
        os.remove(working_path)

# Step 2: Copy the dataset properly
shutil.copytree(input_path, working_path)
print("Dataset copied successfully.")

# Step 3: Verify
print("Contents of working_path:", os.listdir(working_path))

Dataset copied successfully.
Contents of working_path: ['malignant', 'benign', 'normal']


### Remove Segmentation Masks

In [2]:
mask_keywords = ["mask"]
for class_name in os.listdir(working_path):
    class_path = os.path.join(working_path, class_name)
    if not os.path.isdir(class_path):
        continue
    for file_name in os.listdir(class_path):
        if any(keyword in file_name.lower() for keyword in mask_keywords):
            os.remove(os.path.join(class_path, file_name))

In [3]:
classes = [
    c for c in os.listdir(working_path)
    if os.path.isdir(os.path.join(working_path, c))
]

print(f"Classes found: {classes}")

for cls in classes:
    path = os.path.join(working_path, cls)
    count = len([
        f for f in os.listdir(path)
        if os.path.isfile(os.path.join(path, f))
    ])
    print(f"{cls}: {count} images")

Classes found: ['malignant', 'benign', 'normal']
malignant: 210 images
benign: 437 images
normal: 133 images


In [4]:
original_dir = '/kaggle/working/Dataset_BUSI_with_GT'  # path where your class folders are
base_dir = '/kaggle/working/breast_cancer_split'      # new folder for split dataset

train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

classes = ['benign', 'malignant','normal']

for folder in [train_dir, val_dir, test_dir]:
    os.makedirs(folder, exist_ok=True)
    for cls in classes:
        os.makedirs(os.path.join(folder, cls), exist_ok=True)

### Dataset Split

In [5]:
import os
import shutil
import random
split_ratio = (0.7, 0.15, 0.15)  # train, val, test

# Split dataset
for cls in classes:
    cls_path = os.path.join(original_dir, cls)
    images = os.listdir(cls_path)
    random.shuffle(images)

    total = len(images)
    train_end = int(split_ratio[0] * total)
    val_end = train_end + int(split_ratio[1] * total)

    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    for img in train_images:
        shutil.copy(os.path.join(cls_path, img),
                    os.path.join(train_dir, cls, img))

    for img in val_images:
        shutil.copy(os.path.join(cls_path, img),
                    os.path.join(val_dir, cls, img))

    for img in test_images:
        shutil.copy(os.path.join(cls_path, img),
                    os.path.join(test_dir, cls, img))


In [6]:
for folder in ['train', 'val', 'test']:
    print(f"\n{folder.upper()}")
    for cls in classes:
        path = os.path.join(base_dir, folder, cls)
        print(cls, ":", len(os.listdir(path)))


TRAIN
benign : 305
malignant : 147
normal : 93

VAL
benign : 65
malignant : 31
normal : 19

TEST
benign : 67
malignant : 32
normal : 21


In [7]:
import tensorflow as tf
tf.keras.backend.clear_session()

2026-03-18 22:20:01.411423: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773872401.854641      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773872401.968648      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773872402.977354      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773872402.977391      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773872402.977394      55 computation_placer.cc:177] computation placer alr

###  Data Augmentation

In [8]:
import tensorflow as tf
import numpy as np 
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

IMG_SIZE = 224
BATCH_SIZE = 16
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
os.environ['TF_DETERMINISTIC_OPS'] = '1'      
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
tf.random.set_seed(seed)
np.random.seed(seed) 
random.seed(seed)


train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    
    zoom_range=0.2,
    shear_range=0.1,
    
    horizontal_flip=True,
    
    brightness_range=[0.8,1.2],
    
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=seed

)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=seed
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 545 images belonging to 3 classes.
Found 115 images belonging to 3 classes.
Found 120 images belonging to 3 classes.


### Model Architecture

In [9]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models

base_model = ResNet50(
    weights="imagenet",   
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=[
    'accuracy'
])

I0000 00:00:1773872440.683845      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773872440.690045      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


### Class Balancing

In [10]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_labels = train_generator.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(class_labels),
    y=class_labels
)

class_weights_dict = dict(enumerate(class_weights))
print(class_weights_dict)
early_stop = EarlyStopping(
    monitor='val_loss',        
    patience=5, 
    restore_best_weights=True, 
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath='/kaggle/working/best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)


{0: np.float64(0.5956284153005464), 1: np.float64(1.2358276643990929), 2: np.float64(1.9534050179211468)}


### Initial Training

In [11]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    class_weight=class_weights_dict,
     callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30


I0000 00:00:1773872452.158020     143 cuda_dnn.cc:529] Loaded cuDNN version 91002


35/35 ━━━━━━━━━━━━━━━━━━━━ 26s 491ms/step - accuracy: 0.3419 - loss: 1.5287 - val_accuracy: 0.6870 - val_loss: 0.7081
Epoch 2/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 413ms/step - accuracy: 0.5696 - loss: 1.1272 - val_accuracy: 0.7304 - val_loss: 0.6285
Epoch 3/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 412ms/step - accuracy: 0.5677 - loss: 0.9783 - val_accuracy: 0.7391 - val_loss: 0.6222
Epoch 4/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 414ms/step - accuracy: 0.6245 - loss: 0.8281 - val_accuracy: 0.7478 - val_loss: 0.6044
Epoch 5/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 409ms/step - accuracy: 0.7117 - loss: 0.6978 - val_accuracy: 0.7913 - val_loss: 0.6177
Epoch 6/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 408ms/step - accuracy: 0.6631 - loss: 0.7911 - val_accuracy: 0.8000 - val_loss: 0.6118
Epoch 7/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 405ms/step - accuracy: 0.6875 - loss: 0.7024 - val_accuracy: 0.7913 - val_loss: 0.6111
Epoch 8/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 413ms/step - accuracy: 0.6838 - loss: 0.6968 - val_accuracy: 0.791

### Initial Test Results

In [12]:
test_generator.reset()
test_loss, test_acc= model.evaluate(test_generator)
print("Test Accuracy:", test_acc)

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 201ms/step - accuracy: 0.8644 - loss: 0.4023
Test Accuracy: 0.8333333134651184


In [13]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
test_generator.reset()
# Predict
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)

# True labels
y_true = test_generator.classes

class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 435ms/step
              precision    recall  f1-score   support

      benign       0.86      0.84      0.85        67
   malignant       0.74      0.78      0.76        32
      normal       0.90      0.90      0.90        21

    accuracy                           0.83       120
   macro avg       0.83      0.84      0.84       120
weighted avg       0.84      0.83      0.83       120



In [14]:
cm = confusion_matrix(y_true, y_pred)

# Print raw numbers
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[56  9  2]
 [ 7 25  0]
 [ 2  0 19]]


### Fine Tunning

In [15]:
for layer in base_model.layers[-30:]:
    layer.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=[
    'accuracy']
)

In [16]:
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    class_weight=class_weights_dict,
    callbacks=[early_stop]
    )

Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 29s 501ms/step - accuracy: 0.6108 - loss: 1.4124 - val_accuracy: 0.7478 - val_loss: 0.6221
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 423ms/step - accuracy: 0.7305 - loss: 0.7993 - val_accuracy: 0.7304 - val_loss: 0.6828
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 406ms/step - accuracy: 0.7717 - loss: 0.5200 - val_accuracy: 0.7478 - val_loss: 0.6585
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 414ms/step - accuracy: 0.7420 - loss: 0.5737 - val_accuracy: 0.7739 - val_loss: 0.5505
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 414ms/step - accuracy: 0.8060 - loss: 0.5083 - val_accuracy: 0.7826 - val_loss: 0.5277
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 15s 417ms/step - accuracy: 0.7993 - loss: 0.4694 - val_accuracy: 0.8174 - val_loss: 0.4688
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 15s 415ms/step - accuracy: 0.8066 - loss: 0.4129 - val_accuracy: 0.8174 - val_loss: 0.4605
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 14s 415ms/step - accuracy: 0.7662 - loss: 0.4530 - val_accu

### Fine-Tuned Results

In [17]:
test_generator.reset()
test_loss, test_acc = model.evaluate(test_generator)
print("Test Accuracy after Fine-Tuning:", test_acc)

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 197ms/step - accuracy: 0.8857 - loss: 0.3360
Test Accuracy after Fine-Tuning: 0.8833333253860474


In [18]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
test_generator.reset()
# Predict
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)

# True labels
y_true = test_generator.classes

class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 439ms/step
              precision    recall  f1-score   support

      benign       0.95      0.85      0.90        67
   malignant       0.76      0.91      0.83        32
      normal       0.91      0.95      0.93        21

    accuracy                           0.88       120
   macro avg       0.87      0.90      0.89       120
weighted avg       0.89      0.88      0.88       120



In [19]:
cm = confusion_matrix(y_true, y_pred)

# Print raw numbers
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[57  9  1]
 [ 2 29  1]
 [ 1  0 20]]


In [ ]:
# # Save in Keras .keras format
model.save('/kaggle/working/breast_cancer_ultrasound_resnet50.keras')

### Pipeline

In [ ]:
import os
import json
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input

IMG_SIZE = 224


def load_breast_ultrasound_model(model_path):
  
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    model = load_model(model_path)
    return model

def preprocess_ultrasound_image(image_path, img_size=IMG_SIZE):
    
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")

    img = Image.open(image_path).convert("RGB")
    img = img.resize((img_size, img_size))
    
    arr = np.array(img, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)   
    arr = preprocess_input(arr)

    return arr

def predict_breast_ultrasound(
    image_path,
    model,
    labels=None): 
    if labels is None:
        labels = {
            0: 'benign',
            1: 'malignant',
            2: 'normal'
        }

    input_tensor = preprocess_ultrasound_image(image_path)

    # predict
    predictions = model.predict(input_tensor, verbose=0)[0]
    predicted_index = int(np.argmax(predictions))
    predicted_class = labels[predicted_index]
    confidence = float(np.max(predictions))

    # probabilities dictionary
    probabilities = {
        labels[i]: float(predictions[i])
        for i in range(len(predictions))
    }

   
    result = {
        "predicted_class": predicted_class,
        "confidence": round(confidence * 100, 2),
        "probabilities": {
            k: round(v * 100, 2) for k, v in probabilities.items()
        }
    }

    return result

if __name__ == "__main__":
    # Paths
    MODEL_PATH = "breast_cancer_resnet50.keras"
    # Load
    model = load_breast_ultrasound_model(MODEL_PATH)

    # Example single image
    test_image = "/kaggle/input/datasets/basmalatharwat2026/breast-malignant/BrEaSt-malignant/case013.png"

    result = predict_breast_ultrasound(
        image_path=test_image,
        model=model,
        
 
    )

    print(json.dumps(result, indent=4))